# C-01 · Loading and Visualizing Field Outputs

This notebook demonstrates how to read Menura's `.npy` field output files and produce basic visualizations.  It assumes you have a completed (or in-progress) Menura run with its `products/` directory accessible.

**Menura output format:** All field data is written as NumPy `.npy` files via the bundled `cnpy` C++ library.  No HDF5 is used.

**MPI decomposition:** Each MPI rank writes its own tile.  For a run with `mpi_nb_proc_y × mpi_nb_proc_z` = 4 × 4 = 16 processes, each field snapshot consists of 16 files that must be assembled into the global domain.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import os

# --- Configure paths and run parameters ---
# Set this to the directory containing 'products/'
RUN_DIR = '../menura'  # adjust as needed
PRODUCTS = os.path.join(RUN_DIR, 'products')

# MPI decomposition (must match parameters.h values)
NB_PROC_Y = 4
NB_PROC_Z = 4

# Grid size (from parameters.h)
LEN_X = 400
LEN_Y = 100
LEN_Z = 100
DX    = 1.25  # cell size in d_i

print(f'Products directory: {PRODUCTS}')
print(f'MPI grid: {NB_PROC_Y} × {NB_PROC_Z} = {NB_PROC_Y*NB_PROC_Z} ranks')

## 1. Read simulation parameters

Menura writes all parameters to `products/parameters.txt` at startup.  We read it to get the exact values used in the run.

In [ ]:
param_file = os.path.join(PRODUCTS, 'parameters.txt')

p = np.recfromtxt(param_file)
params = {t[0].decode('UTF-8'): float(t[1]) for t in p}

LEN_X = int(params['len_x_cst'])
LEN_Y = int(params['len_y_cst'])
LEN_Z = int(params.get('len_z_cst', 1))
DX    = params['dX']
DT    = params['dt']
NB_PROC_Y = int(params.get('mpi_nb_proc_y', params.get('mpi_nb_proc', 1)))
NB_PROC_Z = int(params.get('mpi_nb_proc_z', 1))
RATE_SAVE_FIELD = int(params['rate_save_field_cst'])

print(f'Grid: {LEN_X} × {LEN_Y} × {LEN_Z}  (dX = {DX} d_i)')
print(f'Time step: dt = {DT} / Ω_ci')
print(f'Field snapshots saved every {RATE_SAVE_FIELD} iterations')

## 2. Helper function: load and assemble a field snapshot

Each snapshot is split across `NB_PROC_Y × NB_PROC_Z` files.  We concatenate along the y and z axes.

In [ ]:
def load_field(field_name, iteration, products_dir, nb_y, nb_z, strip_ghosts=True):
    """
    Load a 3D field snapshot assembled from MPI tiles.
    
    Parameters
    ----------
    field_name : str
        e.g. 'dens', 'B', 'curr'
    iteration : int
        Iteration number (must have been saved)
    products_dir : str
        Path to the 'products/' directory
    nb_y, nb_z : int
        Number of MPI ranks in y and z
    strip_ghosts : bool
        If True, remove the 2-cell ghost layer on each boundary
    
    Returns
    -------
    np.ndarray  — assembled field, shape depends on field type
    """
    rows_y = []
    for ry in range(nb_y):
        row_z = []
        for rz in range(nb_z):
            fn = os.path.join(products_dir,
                              f'{field_name}_it{iteration}_rank_{ry}_{rz}.npy')
            tile = np.load(fn)
            row_z.append(tile)
        rows_y.append(np.concatenate(row_z, axis=-1))   # join along z (last axis)
    full = np.concatenate(rows_y, axis=-2)               # join along y (second-to-last)
    
    if strip_ghosts:
        # Each tile has 4 ghost cells total (+2 each side): keep [2:-2] on each spatial axis
        if full.ndim == 3:       # scalar field: [x, y, z]
            full = full[2:-2, 2:-2, 2:-2]
        elif full.ndim == 4:     # vector field: [3, x, y, z]
            full = full[:, 2:-2, 2:-2, 2:-2]
    return full

# Test: load density at the first saved iteration
it = RATE_SAVE_FIELD
dens = load_field('dens', it, PRODUCTS, NB_PROC_Y, NB_PROC_Z)
print(f'dens shape at it={it}: {dens.shape}')  # expect (LEN_X, LEN_Y, LEN_Z)

## 3. Midplane density slice (x–y plane)

Take a midplane slice at `z = LEN_Z // 2`.

In [ ]:
# Coordinate arrays
x = np.arange(dens.shape[0]) * DX   # x in d_i
y = np.arange(dens.shape[1]) * DX   # y in d_i

# Midplane slice in z
iz_mid = dens.shape[2] // 2
dens_xy = dens[:, :, iz_mid]   # shape (LEN_X, LEN_Y)

fig, ax = plt.subplots(figsize=(12, 3))
im = ax.pcolormesh(x, y, dens_xy.T, cmap='plasma', shading='auto')
plt.colorbar(im, ax=ax, label='Ion density (normalised)')
ax.set_xlabel('x [d_i]')
ax.set_ylabel('y [d_i]')
ax.set_title(f'Total ion density — iteration {it}')
ax.set_aspect('equal')
plt.tight_layout()
plt.savefig('dens_xy_midplane.png', dpi=150)
plt.show()

## 4. Magnetic field

The B-field file has shape `[3, LEN_X, LEN_Y, LEN_Z]` — one array per component.

In [ ]:
B = load_field('B', it, PRODUCTS, NB_PROC_Y, NB_PROC_Z)
print(f'B shape: {B.shape}')   # (3, LEN_X, LEN_Y, LEN_Z)

Bx, By, Bz = B[0], B[1], B[2]
B_mag = np.sqrt(Bx**2 + By**2 + Bz**2)  # |B|

fig, axes = plt.subplots(1, 2, figsize=(14, 3))

im0 = axes[0].pcolormesh(x, y, By[:, :, iz_mid].T, cmap='RdBu_r', shading='auto')
plt.colorbar(im0, ax=axes[0], label='By (normalised)')
axes[0].set_title(f'By component — it={it}')
axes[0].set_xlabel('x [d_i]'); axes[0].set_ylabel('y [d_i]')

im1 = axes[1].pcolormesh(x, y, B_mag[:, :, iz_mid].T, cmap='inferno', shading='auto')
plt.colorbar(im1, ax=axes[1], label='|B| (normalised)')
axes[1].set_title(f'|B| — it={it}')
axes[1].set_xlabel('x [d_i]'); axes[1].set_ylabel('y [d_i]')

plt.tight_layout()
plt.savefig('B_midplane.png', dpi=150)
plt.show()

## 5. Time series of field snapshots

Loop over all saved iterations to track the maximum density over time.

In [ ]:
NB_IT_MAX  = int(params['nb_it_max_cst'])
iterations = range(RATE_SAVE_FIELD, NB_IT_MAX + 1, RATE_SAVE_FIELD)

dens_max = []
times    = []

for it_loop in iterations:
    try:
        d = load_field('dens', it_loop, PRODUCTS, NB_PROC_Y, NB_PROC_Z)
        dens_max.append(d.max())
        times.append(it_loop * DT)
    except FileNotFoundError:
        print(f'  Snapshot it={it_loop} not found, stopping.')
        break

plt.figure(figsize=(8, 3))
plt.plot(times, dens_max, marker='o', markersize=3)
plt.xlabel('Time [1/Ω_ci]')
plt.ylabel('Max density (normalised)')
plt.title('Peak ion density vs. time')
plt.tight_layout()
plt.savefig('dens_max_timeseries.png', dpi=150)
plt.show()

## 6. Exercises

1. Modify `load_field` to work for 2D runs (when `LEN_Z = 1`).
2. Plot the `curr` (ion current) magnitude in the midplane for the last saved iteration.
3. Compute `∇×B` numerically to verify it matches the total current `J_tot` saved by Menura.  Use `np.gradient` with `dx=DX`.
4. What does a ring-shaped density enhancement around the obstacle region indicate physically?